# M1 · 从原始数据重算基线

**核心问题：** 55.21 秒是可信的热态统计量，还是挑选出来的单个样本？从两轮固定 B0 实验原始记录重算，严格排除冷启动与冒烟数据。

**学习闭环：** 先预测 → 检查固定证据 → 独立计算 → 批判性解释 → 设计受控实验。

对应理论课：[打开 course/M01_baseline/LESSON.zh-CN.md](../course/M01_baseline/LESSON.zh-CN.md)。切换语言：[English](./01_baseline_and_uncertainty.ipynb)。

**默认模式：** 纯 CPU、无需联网、只读仓库证据。不会自动启动 GPU。


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root (see notebooks/README).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab import notebook_support as ns
import statistics
print("Course repository:", ROOT)
print("Mode: OFFLINE / committed reference evidence; NO GPU work is launched by default.")


## 检查真实数据并独立计算

按照顺序执行；每个比例都要解释分母与适用范围。

In [ ]:
raw = ns.reference_warm("B0")
claimed = ns.summary_rows()["B0"]
observed = statistics.median(raw)
assert len(raw) == claimed["n_measure"] == 18
assert abs(observed - claimed["warm_total_median_s"]) < 0.03
print(f"B0 reference: n={len(raw)}, median={observed:.3f}s, min={min(raw):.3f}s, max={max(raw):.3f}s")
print("Do NOT label these samples as a new physical GPU run.")
plot_values = {"B0 p50": observed, "min": min(raw), "max": max(raw)}
plot_ylabel, plot_title = "seconds", "Measured B0 warm records"


In [ ]:
SHOW_PLOT = False  # Optional: requires matplotlib in the notebook kernel
if SHOW_PLOT:
    import matplotlib.pyplot as plt
    labels = list(plot_values)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(labels, [plot_values[label] for label in labels])
    ax.set_ylabel(plot_ylabel)
    ax.set_title(plot_title)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Optional plot skipped; set SHOW_PLOT=True to visualize the measured values.")


In [ ]:
# Teaching example: deterministic bootstrap of the median (NOT a formal multi-session CI)
import random
rng = random.Random(42)
replicas = sorted(statistics.median(rng.choices(raw, k=len(raw))) for _ in range(500))
lo, hi = replicas[12], replicas[487]  # illustrative 2.5–97.5 percentile indexes
print(f"Illustrative bootstrap median interval: [{lo:.3f}, {hi:.3f}] seconds")
print("Limitation: resampling 18 observations as IID ignores within-session correlation.")


## 可选 GPU 真机冒烟——必须显式授权

需要 ROCm 环境、模型与 GPU 额度。CI 不执行；结果只存于 `runs/`，不得混入官方证据。

In [ ]:
RUN_GPU = False  # Only switch on after checking platform quota + available models
if RUN_GPU:
    # Also set QIR_NOTEBOOK_GPU_ACK=YES in your own kernel environment.
    # The helper enforces --smoke and writes under runs/, never evidence/.
    ns.gpu_smoke("B0", "notebook-b0", confirmed=True)
else:
    print("GPU NOT_RUN; evidence analysis above is valid without GPU access.")


## 学生思考 / 自检

只选取两轮中的一轮重新计算 B0 中位数。为什么公布两轮比只展示最好一轮更可信？解释 B0 的 n=18 与其他实验 n=9 的差别。

**证据规范：** 用自己的语言完成回答。本 Notebook 重算的是已提交记录，不是你的 GPU 实测。如果自愿运行 smoke，结果必须放在 `runs/`，不能宣传为正式全量统计。
